# 06 - Similarity Validation

Strict cosine-similarity validation with positive-score filtering, thresholds, type filters, and empty-result safety.


In [ ]:
import pickle
import re
from pathlib import Path

import numpy as np
import pandas as pd
from sklearn.metrics.pairwise import cosine_similarity

DATA_DIR = Path("../data/processed")
df = pd.read_csv(DATA_DIR / "tfidf_dataset.csv")

with open(DATA_DIR / "tfidf_vectorizer.pkl", "rb") as f:
    vectorizer = pickle.load(f)
with open(DATA_DIR / "tfidf_matrix.pkl", "rb") as f:
    tfidf_matrix = pickle.load(f)

print("Data:", df.shape)
print("TF-IDF:", tfidf_matrix.shape)

if tfidf_matrix.shape[0] != len(df):
    raise AssertionError("Dataset and TF-IDF matrix row counts do not match. Re-run 05_tfidf.ipynb.")


In [ ]:
def clean_text(text):
    text = str(text).lower()
    text = re.sub(r"[^a-z0-9\s]", " ", text)
    text = re.sub(r"\s+", " ", text).strip()
    return text


def enrich_keywords(text):
    text = str(text).lower()
    enrichment_map = {
        "pantai": ["beach"],
        "beach": ["pantai"],
        "gunung": ["mountain"],
        "air terjun": ["waterfall"],
        "danau": ["lake"],
        "gua": ["cave"],
        "taman": ["park"],
        "candi": ["temple"],
        "temple": ["candi"],
        "museum": ["museum"],
        "warung": ["food", "kuliner"],
        "cafe": ["coffee", "kuliner"],
        "coffee": ["cafe", "kopi"],
        "kopi": ["coffee", "cafe"],
        "restoran": ["restaurant", "kuliner"],
        "restaurant": ["restoran", "kuliner"],
        "hotel": ["lodging", "penginapan"],
        "penginapan": ["hotel", "lodging"],
        "resort": ["hotel", "lodging"],
        "villa": ["hotel", "lodging"],
        "murah": ["budget", "cheap"],
        "budget": ["murah", "cheap"],
    }
    extra_terms = []
    for keyword, terms in enrichment_map.items():
        if re.search(rf"\b{re.escape(keyword)}\b", text):
            extra_terms.extend(terms)
    return " ".join([text] + extra_terms)


def infer_type_filter(query):
    query_clean = clean_text(query)
    if re.search(r"\b(?:hotel|penginapan|resort|villa)\b", query_clean):
        return "hotel"
    if re.search(r"\b(?:cafe|coffee|kopi|restoran|restaurant|warung|kuliner)\b", query_clean):
        return "kuliner"
    if re.search(r"\b(?:pantai|beach|candi|temple|museum|wisata)\b", query_clean):
        return "wisata"
    return None


def apply_intent_gate(results, query):
    if results.empty:
        return results

    query_clean = clean_text(query)
    profile = (
        results["name"].fillna("") + " " +
        results.get("subtypes_clean", results.get("subtypes", "")).fillna("") + " " +
        results.get("subtypes", "").fillna("")
    ).str.lower()

    if re.search(r"\b(?:pantai|beach)\b", query_clean):
        return results[profile.str.contains(r"\b(?:pantai|beach)\b", regex=True)]
    if re.search(r"\b(?:cafe|coffee|kopi)\b", query_clean):
        return results[profile.str.contains(r"\b(?:cafe|coffee|kopi)\b", regex=True)]
    return results


In [ ]:
def recommend_places(query, df, vectorizer, tfidf_matrix, top_n=5, filter_type=None, min_score=0.1):
    valid_types = {"wisata", "kuliner", "hotel"}
    active_filter = filter_type or infer_type_filter(query)

    if active_filter is not None and active_filter not in valid_types:
        raise ValueError(f"filter_type must be one of {sorted(valid_types)} or None.")

    query_text = clean_text(enrich_keywords(query))
    if not query_text:
        return pd.DataFrame(columns=["name", "rating", "type", "subtypes", "price_estimate", "similarity_score"])

    query_vec = vectorizer.transform([query_text])
    sim_scores = cosine_similarity(query_vec, tfidf_matrix).flatten()
    sim_scores = np.nan_to_num(sim_scores, nan=0.0, posinf=0.0, neginf=0.0)

    results = df.copy()
    results["similarity_score"] = sim_scores

    if active_filter:
        results = results[results["type"] == active_filter]

    results = results[results["similarity_score"] > 0]
    results = results[results["similarity_score"] > min_score]
    results = apply_intent_gate(results, query)
    results = results.sort_values("similarity_score", ascending=False)

    output_columns = ["name", "rating", "type", "subtypes", "price_estimate", "similarity_score"]
    return results[output_columns].head(top_n).reset_index(drop=True)


In [ ]:
pantai_results = recommend_places("pantai", df, vectorizer, tfidf_matrix, top_n=10, min_score=0.1)
cafe_results = recommend_places("cafe", df, vectorizer, tfidf_matrix, top_n=10, min_score=0.1)
hotel_results = recommend_places("hotel murah", df, vectorizer, tfidf_matrix, top_n=10, min_score=0.1)

print("Pantai results")
print(pantai_results[["name", "type", "subtypes", "price_estimate", "similarity_score"]])
print("\nCafe results")
print(cafe_results[["name", "type", "subtypes", "price_estimate", "similarity_score"]])
print("\nHotel murah results")
print(hotel_results[["name", "type", "subtypes", "price_estimate", "similarity_score"]])


In [ ]:
def assert_relevant_only(label, results, expected_type, required_pattern=None):
    if results.empty:
        raise AssertionError(f"{label}: no results returned.")
    if results["similarity_score"].isna().any():
        raise AssertionError(f"{label}: NaN similarity_score found.")
    if not (results["type"] == expected_type).all():
        raise AssertionError(f"{label}: found wrong type: {results['type'].unique()}")
    if not (results["similarity_score"] > 0.1).all():
        raise AssertionError(f"{label}: found score <= 0.1")
    if required_pattern:
        profile = (results["name"].fillna("") + " " + results["subtypes"].fillna("")).str.lower()
        if not profile.str.contains(required_pattern, regex=True).all():
            raise AssertionError(f"{label}: found irrelevant item outside required pattern {required_pattern}")

assert_relevant_only("pantai", pantai_results, "wisata", r"\b(?:pantai|beach)\b")
assert_relevant_only("cafe", cafe_results, "kuliner", r"\b(?:cafe|coffee|kopi)\b")
assert_relevant_only("hotel murah", hotel_results, "hotel")

print("All strict behavior tests passed.")
